In [ ]:
import sys
import os

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.data import get_ibis_connection

In [ ]:
tbl_name = "air_traffic"

postgres_config = {
    "user": "postgres",
    "password": "password",
    "host": "postgres",
    "port": 5432,
    "database": "my_db",
}

con = get_ibis_connection(
    backend="postgres",
    postgres_config=postgres_config,
)

# csv_path = project_root + "/data/air_traffic_gold.csv"
# con = get_ibis_connection(
#     backend="duckdb",
#     duckdb_csv_path=csv_path,
# )

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_template = """
Given the following SQL table, your job is to write queries given a user’s request.
Return just the SQL query as plain text, without additional text, and don't use markdown format. 
Please ensure that the field names in the query are enclosed in double quotes.
CREATE TABLE {tbl_name} ({schema})

""".strip()

user_template = "Write a SQL query that returns: {question}"

messages = [("system", system_template), ("user", user_template)]

prompt_template = ChatPromptTemplate.from_messages(messages)

In [ ]:
from sql_ai_agent.db_handler import get_tbl_attr

tbl_attr = get_tbl_attr(con = con, tbl_name = tbl_name)

schema = tbl_attr.schema

print(schema)

In [ ]:
from langchain_openai import ChatOpenAI

base_url = "https://api.openai.com/v1"
model = "gpt-4o"

api_key = os.getenv("OPENAI_API_KEY")

llm = ChatOpenAI(
  base_url=base_url, 
  api_key=api_key, 
  temperature=0, model=model
  )


In [ ]:
chain = prompt_template | llm


In [ ]:
print(chain)

In [ ]:
question = "How many passengers landed during 2024?"


In [ ]:
llm_output = chain.invoke(
    {
        "question": question,
        "tbl_name": tbl_name,
        "schema": schema,
    }
)


In [ ]:
query = llm_output.content
print(query)


In [ ]:
con.sql(query).execute()

In [ ]:
def basic_sql_agent(chain, question, tbl_name, schema, con):
    llm_output = chain.invoke(
        {
            "question": question,
            "tbl_name": tbl_name,
            "schema": schema,
        }
    )
    query = llm_output.content
    print("The return SQL query:")
    print("_" * 60)
    print(query)
    print("_"* 60)
    output = con.sql(query).execute()
    return output


In [ ]:
basic_sql_agent(
    chain,
    question="how many passengers landed during 2024?",
    tbl_name=tbl_name,
    schema=schema,
    con=con,
)
